In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
import default_risk.config as cfg
from default_risk.scripts.auxiliar_eda_function import check_invariant
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import dtale
import logging

log = logging.getLogger('werkzeug')

bureau_df= pd.read_csv(cfg.BUREAU)

data_frame_size=len(bureau_df)

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

In [ ]:
print(data_frame_size)

Invariants found at the moment: (# number of cell with the proofs and relevant code)

1- The factical enndate always point to the past: DAYS_CREDIT_ENDDATE < 0 (100%) #5


Soft constraints:

1- if have a factical end date defined ("DAYS_ENDDATE_FACT" != null), the loan is not active ("CREDICT_ACTIVE" != "Active") (99.9%) #3
Anomalies: Potential data corruption. 

2- if the contract are marked as closed ("CREDICT_ACTIVE" == "Closed") then the conctract have defined a factical end date ("DAYS_ENDDATE_FACT" != null) (99.99%)

Decisions summary: 

1- In cases where the soft constraint #1 is violated, we will input the status as "Closed" taking the DAYS_ENDDATE_FACT as source of true.

2- In cases where the soft contraint #2 we will drop the observations asumming data corruption.

3- For the column "DAYS_CREDIT_ENDDATE" we will segmentate it flags and a continous scale. #8
    a- From -2850 to 1600 lets keep the continous scale as variable.
    b- For values bellow -2850 we will input NaN. (176 observations)
    c- from 1600 to 12000 we will cut, create the flag "positive_first_cluster" and keep that segment of the scale in other variable.
    d- from 12000 to 18000, create the flag "second_positive_cluster" and remove from the original scale 
    d- from 27000 to 28500, create the flag "third_positive_cluster" and remove from the original scale
    e- from 30000 to 31500, create the flag "fourth_positive_cluster and remove from the original scale

4- For DAYS_CREDIT_UPDATE we will clip the scale at -3000 for the negative side of the scale, and in 0 for the positive side (no positive values expected) #9

In [ ]:
#1
#create the files por data data dictionary
create_files_nulls_per_colmun(bureau_df,"bureu")

In [ ]:
#2
#run the screening script on bureau
eda_per_table_printing_results(bureau_df, schema, "bureau",False)

In [ ]:
#3
check_invariant((bureau_df["DAYS_ENDDATE_FACT"] > 0),"the factical enndate is positive (future date)",data_frame_size)

active_mask= (bureau_df["CREDIT_ACTIVE"] == "Active")
have_endate_mask= (bureau_df["DAYS_ENDDATE_FACT"].notna())
check_invariant((active_mask & have_endate_mask),"we have defined a factical date of end of contract but it's flagged as active",data_frame_size)

is_closed_mask= (bureau_df["CREDIT_ACTIVE"] == "Closed")
check_invariant((is_closed_mask & (~have_endate_mask)),"where are closed without factical endate",data_frame_size)

In [ ]:
#4
#in order to understand the nulls in "AMT_ANNUITY"
bureau_prev_contract_without_annuity= bureau_df[bureau_df["AMT_ANNUITY"].isnull()]
len(bureau_prev_contract_without_annuity)
rows_to_analyze=bureau_prev_contract_without_annuity[bureau_prev_contract_without_annuity["CREDIT_ACTIVE"] == "Active"]
print(rows_to_analyze["CREDIT_TYPE"].value_counts())
#seems like active loans of all kind can have AMT_ANNUITY as missing value. 
#This suggest that missing values are more correlated with the way the data are gathering than with the nature of the loans in the sample.

In [ ]:
#5
#in order to understand the missing values in "DAYS_ENDDATE_FACT"
without_endate= bureau_df[bureau_df["DAYS_ENDDATE_FACT"].isnull()]
len(without_endate)
print("status of observations with enndate in null:")
print(without_endate["CREDIT_ACTIVE"].value_counts())
print("status of all the dataset:")
print(bureau_df["CREDIT_ACTIVE"].value_counts())
#we have 125 rows without endate_fact but marked as closed. 



In [ ]:
#6
have_no_limit_card_mask= (bureau_df["AMT_CREDIT_SUM_LIMIT"].isnull()) 
credit_card_loan_mask= (bureau_df["CREDIT_TYPE"] == "Credit card")
is_active_mask= (bureau_df["CREDIT_ACTIVE"] == "Active")
rows_to_analyze= bureau_df[have_no_limit_card_mask & credit_card_loan_mask & is_active_mask]


print("we have " + str((credit_card_loan_mask & is_active_mask).sum()) + " active credit card loans")
print("where " + str(len(rows_to_analyze)) + " have no limit defined")

dtale.show(rows_to_analyze)
#this observation don't let clear the reason to exist credit card loans without limit but was useful to realize there is loans like 
#SK_ID_BUREAU = 5714465 where AMT_CREDIT_SUM is 0 and AMT_CREDIT_SUM_DEBT == 0 or nan at the same time. And with those 2 fields without providing info we don't have how to figured out the ammount of 
#the debt. This expose the need of define a minimun ammount of info provided for a observation to be considered in this table due the poor quality of the data.

In [ ]:
#looking for sentinel values or some artificial discretization in the variable.
#7
column_to_analyze= bureau_df["DAYS_CREDIT"]

counts = (
    column_to_analyze
    .abs()
    .value_counts()
    .sort_index()
)
plt.figure(figsize=(14, 6))

plt.scatter(
    counts.index,
    counts.values,
    s=5
)

plt.title("Occurrences per Exact DAYS_CREDIT Value")

plt.xlabel("Absolute DAYS_CREDIT")
plt.ylabel("Occurrences")

plt.show()

In [ ]:

#In the stage of variable profiling we discovered several outliers and values that appear implausible given the variable description.
#Nevertheless, the pattern does not match the classic sentinel values, because there are not only a repeated values. So in order to understand these values let's visualize the scale. 
column_to_analyze= bureau_df["DAYS_CREDIT_ENDDATE"]

counts = (
    column_to_analyze
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(40,6))

ax.scatter(
    counts.index,
    counts.values,
    s=5
)



ax.xaxis.set_major_locator(ticker.MultipleLocator(1500))
ax.set_title("Occurrences per Exact DAYS_CREDIT_ENDDATE Value")
ax.set_xlabel("DAYS_CREDIT_ENDDATE")
ax.set_ylabel("Occurrences")

print((column_to_analyze < -2850).sum())

print((column_to_analyze == 12000).sum())

print(((column_to_analyze > 12000) & (column_to_analyze < 18000)).sum())


plt.show()

#this shows clear clusters and discrete jumps that can be produced in several ways. Regardless, we have the center of the graphic that exhibits the expected behavior for type of variable. So in order 
#to preserve the scale of the variable and avoid mixing what seems to be different scales or phenomena, let's split the scale to separate the clusters, and only keep as a continous variable the center that has a coherent distribution.
#we can recognize clear cuts active_mask -2900 and 1600 for the center of the scale, and different clusters with more or fewer observations.

In [ ]:
#9
column_to_analyze= bureau_df["DAYS_CREDIT_UPDATE"]

counts = (
    column_to_analyze
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(30,6))

ax.scatter(
    counts.index,
    counts.values,
    s=5
)

#checking cutting points.
print((column_to_analyze == -3000).sum())
print((column_to_analyze < -3000).sum())
print((column_to_analyze == 0).sum())
print((column_to_analyze > 0).sum())



ax.xaxis.set_major_locator(ticker.MultipleLocator(1500))
ax.set_title("Occurrences per Exact DAYS_CREDIT_UPDATE Value")
ax.set_xlabel("DAYS_CREDIT_UPDATE")
ax.set_ylabel("Occurrences")
